# Compiler Design Lab: Learning Lex / Flex in Google Colab

This is Exp 5 in your Compiler Design Lab. This notebook is a self-contained lab for learning **Flex** (Fast Lexical Analyzer Generator), the standard tool for building lexical analyzers / scanners.

**What you will do in this notebook:**

1. Install Flex in Colab and confirm it works.
2. Work through 7 short Flex programs.
3. Read the manual under each example .
4. "task of the day" exercises on your own.

**The Flex workflow** (same every time):

```
file.l  --(flex)-->  lex.yy.c  --(gcc)-->  executable  --(run with input)-->  output
```

Every code cell below is independent — run them top to bottom the first time, then feel free to re-run any single cell while you experiment.


## 1. Installing Flex in Colab

Colab's runtime is a full Ubuntu machine, and Flex isn't installed by default. Install it with `apt-get` (Colab notebooks run as root, so no `sudo` is needed).


In [ ]:
!apt-get update -y > /dev/null
!apt-get install -y flex bison > /dev/null
!echo "Install step finished."


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Install step finished.


Check that both tools are on the `PATH` and report a version. If you see version numbers below (not "command not found"), the install worked.


In [ ]:
!flex --version
!bison --version | head -1
!gcc --version | head -1


flex 2.6.4
bison (GNU Bison) 3.8.2
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0


### Sample  test

Write a two-line Flex program with `%%writefile`, generate the scanner, compile it, and run it against sample text typed inline with `echo`.

If this cell prints the two matched greetings below, Flex is fully working in this Colab runtime.


In [2]:
!pwd
!ls

/content
sample_data


In [3]:
%%writefile test1.l
%{
#include <stdio.h>
%}

%%
"hello"     { printf("Greeting found: %s\n", yytext); }
"world"     { printf("Place found: %s\n", yytext); }
\n          { /* ignore newlines */ }
.           { /* ignore everything else */ }
%%

int yywrap(void) { return 1; }

int main(void) {
    yylex();
    return 0;
}


Writing test1.l


In [4]:
!ls

sample_data  test1.l


In [ ]:
!flex test1.l

In [ ]:
!pwd

/content


In [ ]:
!ls

lex.yy.c  sample_data  test1.l


In [ ]:
!flex test1.l
!gcc lex.yy.c -o test -lfl
!echo "hello world, this is flex" | ./test


Greeting found: hello
Place found: world


**Expected output:**

```
Greeting found: hello
Place found: world
```

**What just happened, step by step:**

- `flex test.l` read the `.l` specification and generated `lex.yy.c` — plain C source containing the scanner.
- `gcc lex.yy.c -o test -lfl` compiled that C file into an executable named `test`. `-lfl` links Flex's runtime library (it supplies `yywrap` on some systems and helper routines; we still define our own `yywrap` here, which is fine).
- `echo "..." | ./test` piped text into the program's `stdin`, which is exactly what `yylex()` reads from by default.

You now have a working Flex toolchain. The same three commands (`flex`, `gcc ... -lfl`, run) repeat for every example below — only the `.l` file changes.


## 2. Example Programs

Each example below is a complete, runnable Flex program. Every one follows the same three-cell pattern:

1. `%%writefile exN_name.l` — the Flex specification.
2. `!flex ... && gcc ... && run` — build and run it against sample input.
3. A manual explaining the aim, the concepts introduced, and how each rule works.

Flex files have three sections separated by `%%`:

```
definitions   (C code in %{ %}, plus named patterns)
%%
rules         (pattern { action })
%%
user code     (main(), helper functions)
```


### Example 1 — Minimal Flex program (structure of a `.l` file)

**Aim:** Understand the three-part structure of a Flex file and how a rule fires when its pattern matches.

**Concepts introduced:** `%{ %}` declarations block, the rules section, `yytext` (the matched lexeme), the catch-all `.` pattern, `yywrap()`, `main()` calling `yylex()`.

**1. yylex()**

yylex() is the main scanner (lexer) function generated automatically by Lex/Flex.

**What does it do?**

It reads the input character by character, matches it against the patterns you defined in the Lex file, and executes the corresponding actions.

**2. yywrap()**

When yylex() reaches the end of the input file (EOF), it calls another function named yywrap().

In [ ]:
%%writefile ex1_hello.l
%{
/* Example 1: Minimal Flex program - recognize the word "hello" */
#include <stdio.h>
%}

%%
"hello"     { printf("Greeting found: %s\n", yytext); }
"world"     { printf("Place found: %s\n", yytext); }
\n          { /* ignore newlines */ }
.           { /* ignore everything else */ }
%%

int yywrap(void) { return 1; }

int main(void) {
    yylex();
    return 0;
}


Writing ex1_hello.l


In [ ]:
!flex ex1_hello.l
!gcc lex.yy.c -o ex1_hello -lfl
!echo "hello world, this is flex" | ./ex1_hello


**Sample output:**

```
Greeting found: hello
Place found: world
```

**Manual — how it works:**

- `"hello"` and `"world"` are literal-string patterns. Whenever the scanner's input matches one exactly, the matched text is copied into the global `char *yytext`, and the action in `{ }` runs.
- `\n` matches a newline; the action is empty, so newlines are silently consumed.
- `.` matches any single character not already matched above (Flex tries rules in the order written, and prefers the longest match — `.` is the fallback for everything else, one character at a time).
- `yylex()` is the function Flex generates from your rules. Calling it in `main()` starts scanning `stdin` and keeps consuming input until end-of-file.
- `yywrap()` is called when Flex reaches end-of-input; returning `1` tells it "no more input follows, stop".


### Example 2 — Counting characters, words and lines (like `wc`)

**Aim:**  Count characters, words and lines of an input strings.

**Concepts introduced:** counter variables declared in the definitions section, `yyleng` (length of the matched text), matching runs of non-whitespace as a single "word" token.


In [ ]:
%%writefile ex2_wc.l
%{
/* Example 2: Count characters, words and lines - like the "wc" command */
#include <stdio.h>
int char_count = 0, word_count = 0, line_count = 0;
%}

%%
\n          { line_count++; char_count++; }
[^ \t\n]+   { word_count++; char_count += yyleng; }
.           { char_count++; }
%%

int yywrap(void) { return 1; }

int main(void) {
    yylex();
    printf("Characters: %d\n", char_count);
    printf("Words     : %d\n", word_count);
    printf("Lines     : %d\n", line_count);
    return 0;
}


Writing ex2_wc.l


In [ ]:
!flex ex2_wc.l
!gcc lex.yy.c -o ex2_wc -lfl
!printf "Flex is fun\nWe are learning lex\n" | ./ex2_wc


Characters: 32
Words     : 7
Lines     : 2


**Sample output:**

```
Characters: 32
Words     : 7
Lines     : 2
```

**Manual — how it works:**

- `[^ \t\n]+` is a character class negated with `^` — "one or more characters that are *not* space, tab, or newline". This is how an entire word is matched as a single token, not one character at a time.
- `yyleng` is an integer Flex maintains automatically — the length of whatever `yytext` currently holds. Using it avoids calling `strlen(yytext)` yourself.
- Rule order matters here only where patterns could both match; Flex resolves overlaps by preferring the *longest* match, so `[^ \t\n]+` beats `.` for a whole word.
- The three counters live in the definitions section (`%{ %}`) so they're ordinary global C variables, visible to every rule's action and to `main()`.


### Example 3 — Classifying characters (digits, letters, spaces, others)

**Aim:** Write a flex program to  classify input into digits, letters, spaces and other characters

**Concepts introduced:** POSIX-style character classes `[0-9]`, `[a-zA-Z]`, `[ \t\n]`, and how a trailing `.` rule catches everything else (punctuation, symbols).


In [ ]:
%%writefile ex3_classify.l
%{
/* Example 3: Classify input into digits, letters, spaces and other characters */
#include <stdio.h>
int digits = 0, letters = 0, spaces = 0, others = 0;
%}

%%
[0-9]       { digits++; }
[a-zA-Z]    { letters++; }
[ \t\n]     { spaces++; }
.           { others++; }
%%

int yywrap(void) { return 1; }

int main(void) {
    yylex();
    printf("Digits  : %d\n", digits);
    printf("Letters : %d\n", letters);
    printf("Spaces  : %d\n", spaces);
    printf("Others  : %d\n", others);
    return 0;
}


Writing ex3_classify.l


In [ ]:
!flex ex3_classify.l
!gcc lex.yy.c -o ex3_classify -lfl
!printf 'Room 101, price = $45.50!\n' | ./ex3_classify


Digits  : 7
Letters : 9
Spaces  : 5
Others  : 5


**Sample output:**

```
Digits  : 7
Letters : 9
Spaces  : 5
Others  : 5
```

**Manual — how it works:**

- Each rule here matches exactly **one** character (no `+`), so the scanner processes the input one character at a time, classifying each into exactly one bucket.
- `[0-9]`, `[a-zA-Z]`, and `[ \t\n]` are disjoint classes — no character can match more than one, so there's no ambiguity to resolve.
- `.` (matches any character except newline) is placed last and catches punctuation and symbols like `,`, `=`, `$`, `.`, `!` — anything not already claimed by the rules above it.
- This example is the building block for a real lexical analyzer: instead of counting, you'd swap each action for "emit a token".


### Example 4 — Recognizing keywords vs. identifiers

**Aim:** Build a small lexical analyzer that tells C keywords apart from identifiers — the same problem a real compiler's front end solves first.

**Concepts introduced:** named patterns declared in the definitions section (`DIGIT`, `LETTER`), alternation `|` to list literal keywords, rule-order as the tie-breaker between a keyword and the general identifier pattern.


In [ ]:
%%writefile ex4_keyword.l
%{
/* Example 4: Recognize C keywords vs identifiers */
#include <stdio.h>
%}

DIGIT   [0-9]
LETTER  [a-zA-Z_]

%%
"int"|"float"|"char"|"double"|"void"|"return"|"if"|"else"|"while"|"for"   { printf("%-12s KEYWORD\n", yytext); }
{LETTER}({LETTER}|{DIGIT})*   { printf("%-12s IDENTIFIER\n", yytext); }
[ \t\n]     { /* skip whitespace */ }
.           { /* skip other characters for this example */ }
%%

int yywrap(void) { return 1; }

int main(void) {
    yylex();
    return 0;
}


In [ ]:
!flex ex4_keyword.l
!gcc lex.yy.c -o ex4_keyword -lfl
!echo "int num1, num2, sum; float avg;" | ./ex4_keyword


**Sample output:**

```
int          KEYWORD
num1         IDENTIFIER
num2         IDENTIFIER
sum          IDENTIFIER
float        KEYWORD
avg          IDENTIFIER
```

**Manual — how it works:**

- `{LETTER}` and `{DIGIT}` are *named patterns* — defined once above the first `%%`, then reused inside `{ }` braces in the rules section. This keeps regular expressions readable.
- The keyword rule is listed **before** the identifier rule. When both patterns could match the same text (e.g. `int` matches both the literal `"int"` and the general identifier pattern), Flex breaks the tie by **rule order**, not just longest match — so keywords must always be listed above the generic identifier rule, or every keyword would be reported as an identifier instead.
- `{LETTER}({LETTER}|{DIGIT})*` is exactly the classic identifier definition: a letter or underscore, followed by zero or more letters/digits/underscores — this is the same automaton (start on a letter, loop on letter-or-digit) used in lexical-analysis theory.


### Example 5 — Tokenizing an arithmetic expression

**Aim:** Recognize multiple token categories in one scanner: numbers (including decimals), operators, and parentheses.

**Concepts introduced:** matching a number with an optional fractional part `(\.[0-9]+)?`, grouping alternatives for single-character operators, reporting a distinct token name per category.


In [ ]:
%%writefile ex5_tokenize.l
%{
/* Example 5: Tokenize an arithmetic expression into numbers, operators, parentheses */
#include <stdio.h>
%}

%%
[0-9]+(\.[0-9]+)?   { printf("%-12s NUMBER\n", yytext); }
"+"|"-"|"*"|"/"     { printf("%-12s OPERATOR\n", yytext); }
"("                 { printf("%-12s LPAREN\n", yytext); }
")"                 { printf("%-12s RPAREN\n", yytext); }
[ \t\n]             { /* skip whitespace */ }
.                   { printf("%-12s UNKNOWN\n", yytext); }
%%

int yywrap(void) { return 1; }

int main(void) {
    yylex();
    return 0;
}


In [ ]:
!flex ex5_tokenize.l
!gcc lex.yy.c -o ex5_tokenize -lfl
!echo "(12 + 3.5) * 20 / 4" | ./ex5_tokenize


**Sample output:**

```
(            LPAREN
12           NUMBER
+            OPERATOR
3.5          NUMBER
)            RPAREN
*            OPERATOR
20           NUMBER
/            OPERATOR
4            NUMBER
```

**Manual — how it works:**

- `[0-9]+(\.[0-9]+)?` reads as: one or more digits, then *optionally* a literal dot followed by one or more digits. The `?` makes the whole group optional, so both `20` and `3.5` are matched by the same rule.
- `"+"|"-"|"*"|"/"` lists four single-character alternatives in one rule so they all map to the same `OPERATOR` action; you could instead give each its own rule if you needed to distinguish them.
- The final `.` rule labels anything unexpected as `UNKNOWN` — useful while debugging a grammar, since it surfaces characters your pattern set forgot to handle instead of silently ignoring them.


### Example 6 — Case conversion (lowercase to uppercase)

**Aim:** Use Flex to transform text rather than only classify or count it.

**Concepts introduced:** acting on a single matched character with `yytext[0]`, using standard C library functions (`toupper`) inside an action, passing everything else through unchanged with `putchar`.


In [ ]:
%%writefile ex6_upper.l
%{
/* Example 6: Convert lowercase letters in the input to uppercase */
#include <stdio.h>
#include <ctype.h>
%}

%%
[a-z]   { putchar(toupper(yytext[0])); }
.|\n    { putchar(yytext[0]); }
%%

int yywrap(void) { return 1; }

int main(void) {
    yylex();
    return 0;
}


In [ ]:
!flex ex6_upper.l
!gcc lex.yy.c -o ex6_upper -lfl
!echo "Flex makes Lexical Analysis easy, 2026!" | ./ex6_upper


**Sample output:**

```
FLEX MAKES LEXICAL ANALYSIS EASY, 2026!
```

**Manual — how it works:**

- `[a-z]` matches exactly one lowercase letter; its action converts it with `toupper()` and writes it straight to `stdout` with `putchar` — nothing is buffered or reassembled, each character is handled and emitted independently.
- `.|\n` (anything else, including newline) is written back out unchanged, so digits, punctuation, spaces, and already-uppercase letters pass through untouched.
- This pattern — "transform one class of input, pass everything else through" — is the same shape you'd use for tasks like stripping accents, escaping HTML characters, or redacting digits.


### Example 7 — Removing comments from C code (start conditions)

**Aim:** Introduce **start conditions**, Flex's mechanism for giving the scanner "modes" — essential once a language has multi-character constructs like comments that span from a start marker to an end marker.

**Concepts introduced:** `%x` (exclusive start condition), `BEGIN(...)` to switch modes, `<STATE>` prefixes to make a rule apply only in a given mode.


In [ ]:
%%writefile ex7_comments.l
%{
/* Example 7: strip line comments and block comments from C source code */
#include <stdio.h>
%}

%x LINE_COMMENT
%x BLOCK_COMMENT

%%
"//"                    { BEGIN(LINE_COMMENT); }
<LINE_COMMENT>\n        { BEGIN(INITIAL); putchar('\n'); }
<LINE_COMMENT>.         { /* discard */ }

"/*"                    { BEGIN(BLOCK_COMMENT); }
<BLOCK_COMMENT>"*/"     { BEGIN(INITIAL); }
<BLOCK_COMMENT>\n       { putchar('\n'); /* keep line breaks for readability */ }
<BLOCK_COMMENT>.        { /* discard */ }

.|\n                    { putchar(yytext[0]); }
%%

int yywrap(void) { return 1; }

int main(void) {
    yylex();
    return 0;
}


In [ ]:
%%writefile sample.c
#include <stdio.h>   // header file
/* This program
   prints a message */
int main(void) {
    printf("Hello"); // print greeting
    return 0;
}


In [ ]:
!flex ex7_comments.l
!gcc lex.yy.c -o ex7_comments -lfl
!./ex7_comments < sample.c


**Sample output:**

```
#include <stdio.h>


int main(void) {
    printf("Hello");
    return 0;
}
```

(Both comments are gone; blank lines remain where they were, so line numbers in the rest of the file are preserved.)

**Manual — how it works:**

- `%x LINE_COMMENT` and `%x BLOCK_COMMENT` declare two *exclusive* start conditions — extra scanner "modes" besides the default `INITIAL` mode. Exclusive means: while inside one of these modes, **only** rules prefixed with `<THAT_MODE>` are considered; all `INITIAL`-mode rules are ignored until the scanner leaves the mode.
- Matching `"//"` calls `BEGIN(LINE_COMMENT)`, switching modes. From then on, `<LINE_COMMENT>.` silently discards every character until `<LINE_COMMENT>\n` is seen, which prints the newline (so line breaks in the output line up with the original file) and calls `BEGIN(INITIAL)` to return to normal scanning.
- `"/*"` and `"*/"` work the same way for block comments, except the mode is only exited on the literal `*/` marker, so comments spanning multiple lines are handled correctly — that's the entire reason a single `.` catch-all rule (as in earlier examples) isn't enough here; you need a mode that "remembers" you're inside a comment across many characters and lines.
- The final `.|\n` rule (with no `<STATE>` prefix) only applies in `INITIAL` mode, and passes ordinary code straight through unchanged.


## 3. Task of the Day

Now write these yourself. Use the same three-cell pattern (`%%writefile`, then `flex` + `gcc` + run) as the examples above. Don't scroll back and copy an example line-for-line — each task needs at least one new pattern idea that wasn't used above.

**Task 1 — Vowel and consonant counter**
Write a Flex program that reads text and reports the number of vowels and the number of consonants separately (treat both uppercase and lowercase letters). *Hint: you'll want two character-class rules, similar in spirit to Example 3, but split on vowel vs. consonant instead of digit vs. letter.*

**Task 2 — Positive and negative integer counter**
Write a Flex program that reads a line of numbers (e.g. `12 -5 7 -3 0 -9`) and counts how many are positive and how many are negative. *Hint: a number preceded immediately by `-` needs its own pattern — think about how Example 5 matched numbers, and what changes if a minus sign must be included in the match.*

**Task 3 — Token counter for a C statement**
Using the token categories from this notebook (`KEYWORD`, `IDENTIFIER`) plus two new ones — `COMMA` for `,` and `SEMICOLON` for `;` — write a Flex program that reads `int num1, num2, sum;` and prints a running **count** of each token category at the end (not each individual lexeme). *Hint: this is Example 4 with two more rules added, plus counters like Example 2 instead of `printf` inside every rule.*

**Before you start:** decide what your program should do with characters that don't match any of your patterns — silently skip them, or report them as `UNKNOWN`? Either is fine, but be consistent and mention your choice as a comment in your `.l` file.
